# Irodori-TTS v4.1 Small RF · 저장된 LoRA 추론

**GPU 런타임에서 1 → 2 → 3 → 4 → 5번을 실행하세요.** 1번 `lora_path`에 Colab 또는 Google Drive의 **ZIP 파일 / 체크포인트 폴더 경로**를 입력합니다.
학습 데이터와 학습 과정 없이, 저장된 LoRA와 기본 모델로 음성을 생성합니다.

| 입력 종류 | `lora_path` 예시 |
|---|---|
| Colab ZIP | `/content/checkpoint_final_inference.zip` |
| Colab 폴더 | `/content/checkpoint_final` |
| Google Drive ZIP | `/content/drive/MyDrive/checkpoint_final_inference.zip` |
| Google Drive 폴더 | `/content/drive/MyDrive/LoRA/checkpoint_final` |

- 기존 `Irodori_v4_1_RF_Yuina_LoRA_Colab.ipynb`에서 학습한 **Irodori v4.1 Small RF LoRA**용입니다. 다른 기본 모델용 LoRA는 별도 설정이 필요합니다.
- 폴더 안에 `adapter_config.json`과 `adapter_model.safetensors`가 함께 있어야 합니다. ZIP 안의 중첩 폴더도 찾습니다. `.pt` 학습 재개 파일은 복사하지 않습니다.
- 체크포인트가 여러 개면 2번 셀이 후보를 표시합니다. 1번의 `adapter_subdir`에 원하는 상대 경로를 넣고 1~2번을 다시 실행하세요. 빈 값일 때 임의로 고르지 않습니다.
- Google Drive는 해당 경로를 입력한 경우에만 연결합니다. 추론 파일은 `/content/irodori_inference`에 복사하며 원본은 수정하지 않습니다. 다른 셀에서 쓰는 Drive 연결을 유지하기 위해 자동 해제하지 않습니다.
- 5번에서 Gradio 공유 링크가 나옵니다. 기본값은 로그인 없이 접속하며, `enable_login`을 켰을 때만 비밀번호를 입력합니다. 참조 WAV 업로드 → 일본어 문장 입력 → **Generate**로 사용합니다.
- 생성 WAV는 화면에서 다운로드하세요. Colab 런타임이 초기화되면 로컬 복사본과 생성 WAV가 사라집니다.

공식 [Irodori-TTS 소스](https://github.com/Aratako/Irodori-TTS/tree/89f9d8fbd4d51ea019867ee1197725ede1df13c5)와 기존 학습 노트북의 코드·모델 revision을 사용합니다.

In [ ]:
#@title 1. 학습된 LoRA 경로 설정 (ZIP 또는 폴더)
lora_path = "/content/drive/MyDrive/checkpoint_final_inference.zip" #@param {type:"string"}
adapter_subdir = "" #@param {type:"string"}
enable_login = False #@param {type:"boolean"}
gradio_username = "yuina" #@param {type:"string"}
server_port = 7860 #@param {type:"integer"}

from pathlib import Path, PurePosixPath
from contextlib import ExitStack
import getpass, hashlib, json, os, re, shutil, signal, stat, subprocess, tempfile, time
import urllib.error, urllib.request, zipfile

if globals().get('app_proc') is not None and app_proc.poll() is None:
    raise RuntimeError('6번에서 서버를 종료한 뒤 설정을 변경하세요.')
if not lora_path.strip():
    raise ValueError('LoRA 경로를 입력하세요.')
if enable_login and not gradio_username.strip():
    raise ValueError('로그인을 사용할 때는 로그인 ID를 입력하세요.')
if not 1024 <= server_port <= 65535:
    raise ValueError('server_port는 1024~65535 범위로 입력하세요.')

WORK = Path('/content/irodori_inference')
REPO = WORK / 'Irodori-TTS'
DRIVE = Path('/content/drive')
ADAPTER_ROOT = WORK / 'adapters'
MODEL_DIR = WORK / 'models/irodori-v4.1-small-rf'
CODE_REVISION = '89f9d8fbd4d51ea019867ee1197725ede1df13c5'
MODEL_ID = 'Aratako/Irodori-TTS-v4.1-Small'
MODEL_REVISION = '2b28324dc263ed5e6638b3cf3dd94c82ead07b4b'
ADAPTER_FILES = ('adapter_config.json', 'adapter_model.safetensors')
for path in (WORK, ADAPTER_ROOT, MODEL_DIR):
    path.mkdir(parents=True, exist_ok=True)
    if not path.resolve().is_relative_to(Path('/content')) or path.resolve().is_relative_to(DRIVE.resolve()):
        raise ValueError('작업 폴더는 /content의 로컬 폴더여야 합니다.')
os.environ.update({
    'HF_HOME': str(WORK / 'cache/huggingface'),
    'UV_CACHE_DIR': str(WORK / 'cache/uv'),
    'TORCH_HOME': str(WORK / 'cache/torch'),
    'GRADIO_TEMP_DIR': str(WORK / 'gradio_tmp'),
    'GRADIO_ANALYTICS_ENABLED': 'False',
    'TOKENIZERS_PARALLELISM': 'false',
})

def run(args, *, cwd=None):
    process = subprocess.Popen([str(arg) for arg in args], cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        bufsize=1, start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end='')
        if process.wait():
            raise RuntimeError(f'실행 실패 (exit={process.returncode}). 위 로그를 확인하세요.')
    finally:
        stop_process(process)
        process.stdout.close()

def stop_process(process):
    if process is not None and process.poll() is None:
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()

def write_script(name, source):
    path = WORK / name
    path.write_text(source, encoding='utf-8')
    return path

print('LoRA 입력:', lora_path)
print('로컬 작업 폴더:', WORK)

In [ ]:
#@title 2. ZIP / 폴더에서 LoRA 확인 후 로컬 복사
if globals().get('app_proc') is not None and app_proc.poll() is None:
    raise RuntimeError('6번에서 서버를 종료한 뒤 LoRA를 변경하세요.')
ADAPTER = None

def check_relative_path(raw):
    path = PurePosixPath(raw)
    if path.is_absolute() or '..' in path.parts or chr(92) in raw or ':' in raw:
        raise ValueError(f'허용되지 않는 상대 경로: {raw}')
    return path

def choose_adapter(candidates, requested):
    names = sorted(candidates)
    if not names:
        raise ValueError('adapter_config.json과 adapter_model.safetensors가 같은 폴더에 있어야 합니다.')
    if requested.strip():
        selected = check_relative_path(requested.strip()).as_posix()
        if selected not in candidates:
            raise ValueError('adapter_subdir을 확인하세요. 사용 가능한 경로:\n' + '\n'.join(names))
        return selected
    if len(names) != 1:
        raise ValueError('체크포인트가 여러 개입니다. 1번 adapter_subdir에 다음 중 하나를 입력하세요:\n' + '\n'.join(names))
    return names[0]

def check_base_model(metadata):
    expected = dict(repo_id=MODEL_ID, revision=MODEL_REVISION, code_revision=CODE_REVISION)
    if not isinstance(metadata, dict):
        raise ValueError('BASE_MODEL.json 형식이 올바르지 않습니다.')
    for key, value in expected.items():
        if metadata.get(key) and metadata[key] != value:
            raise ValueError(f'학습 당시 기본 모델/코드와 다릅니다: {key}={metadata[key]}. 해당 학습 설정에 맞는 노트북을 사용하세요.')

source = Path(lora_path.strip()).expanduser()
if not source.is_absolute():
    raise ValueError('/content/... 형태의 전체 경로를 입력하세요.')
if source.is_relative_to(DRIVE):
    from google.colab import drive
    drive.mount(str(DRIVE))
source = source.resolve()
if not source.exists():
    raise FileNotFoundError(f'경로가 없습니다: {source}')

# ZIP 전체를 풀지 않고 선택된 어댑터 두 파일만 복사합니다.
with ExitStack() as stack:
    if source.is_file():
        if not zipfile.is_zipfile(source):
            raise ValueError('파일 입력은 ZIP이어야 합니다. 웨이트 단일 파일 대신 체크포인트 폴더를 지정하세요.')
        archive = stack.enter_context(zipfile.ZipFile(source))
        entries = {}
        for info in archive.infolist():
            path = check_relative_path(info.filename)
            mode = info.external_attr >> 16
            if stat.S_ISLNK(mode):
                raise ValueError(f'ZIP의 심볼릭 링크는 지원하지 않습니다: {info.filename}')
            key = path.as_posix()
            if key in entries:
                raise ValueError(f'ZIP의 중복 경로: {key}')
            entries[key] = info
        file_entries = {key: value for key, value in entries.items() if not value.is_dir()}
        candidates = {}
        for key in file_entries:
            path = PurePosixPath(key)
            if path.name == ADAPTER_FILES[0]:
                members = [(path.parent / name).as_posix() for name in ADAPTER_FILES]
                if all(member in file_entries for member in members):
                    candidates[path.parent.as_posix()] = members
        selected = choose_adapter(candidates, adapter_subdir)
        members = candidates[selected]
        sizes = [file_entries[member].file_size for member in members]
        metadata_paths = {(PurePosixPath(selected) / 'BASE_MODEL.json').as_posix(), 'BASE_MODEL.json'}
        metadata_found = False
        for name in sorted(metadata_paths):
            if name in file_entries:
                if file_entries[name].file_size > 1024 * 1024:
                    raise ValueError('BASE_MODEL.json이 너무 큽니다.')
                check_base_model(json.loads(archive.read(file_entries[name]).decode('utf-8-sig')))
                metadata_found = True
        def open_member(member):
            return archive.open(file_entries[member])
    elif source.is_dir():
        candidates = {}
        for folder, dirs, files in os.walk(source, followlinks=False):
            dirs[:] = [name for name in dirs if not (Path(folder) / name).is_symlink()]
            if all(name in files for name in ADAPTER_FILES):
                paths = [Path(folder) / name for name in ADAPTER_FILES]
                if any(path.is_symlink() or not path.resolve().is_relative_to(source) for path in paths):
                    raise ValueError('어댑터 파일의 심볼릭 링크는 지원하지 않습니다.')
                candidates[Path(folder).relative_to(source).as_posix()] = paths
        selected = choose_adapter(candidates, adapter_subdir)
        members = candidates[selected]
        sizes = [path.stat().st_size for path in members]
        metadata_found = False
        for path in sorted({source / 'BASE_MODEL.json', members[0].parent / 'BASE_MODEL.json'}):
            if path.is_file():
                if path.is_symlink() or path.stat().st_size > 1024 * 1024:
                    raise ValueError('BASE_MODEL.json 경로 또는 크기를 확인하세요.')
                check_base_model(json.loads(path.read_text(encoding='utf-8-sig')))
                metadata_found = True
        def open_member(member):
            return member.open('rb')
    else:
        raise ValueError('일반 ZIP 파일 또는 폴더 경로를 입력하세요.')

    if min(sizes) <= 0 or sizes[0] > 1024 * 1024:
        raise ValueError('어댑터 파일이 비어 있거나 설정 JSON이 너무 큽니다.')
    if sum(sizes) + 1024**3 > shutil.disk_usage(WORK).free:
        raise RuntimeError('어댑터 복사 후 여유 공간 1 GiB를 확보할 수 없습니다.')
    # 고유 폴더에 복사하므로 기존 어댑터를 덮어쓰지 않습니다.
    staged = Path(tempfile.mkdtemp(prefix='adapter_', dir=ADAPTER_ROOT))
    for member, name, expected_size in zip(members, ADAPTER_FILES, sizes):
        with open_member(member) as reader, (staged / name).open('wb') as writer:
            shutil.copyfileobj(reader, writer, length=8 * 1024 * 1024)
        if (staged / name).stat().st_size != expected_size:
            raise RuntimeError('복사 도중 원본 파일 크기가 변경되었습니다. 저장 완료 후 다시 실행하세요.')
    config = json.loads((staged / ADAPTER_FILES[0]).read_text(encoding='utf-8-sig'))
    if not isinstance(config, dict) or not config:
        raise ValueError('adapter_config.json은 비어 있지 않은 JSON 객체여야 합니다.')
    ADAPTER = staged

print('선택한 체크포인트:', selected)
print('추론용 로컬 폴더:', ADAPTER)
if not metadata_found:
    print('BASE_MODEL.json이 없습니다. Irodori v4.1 Small RF로 학습한 LoRA인지 확인하세요.')
print('원본 유지. 추론용 두 파일 복사 완료.')

In [ ]:
#@title 3. 공식 의존성 설치 (첫 실행은 수 분 걸립니다)
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('런타임 → 런타임 유형 변경 → GPU를 선택하세요.')
run(['nvidia-smi'])
run(['apt-get', '-qq', 'update'])
run(['apt-get', '-qq', 'install', '-y', 'ffmpeg', 'libsndfile1', 'git', 'build-essential'])
uv = shutil.which('uv') or str(Path.home() / '.local/bin/uv')
if not Path(uv).exists():
    installer = WORK / 'uv-install.sh'
    urllib.request.urlretrieve('https://astral.sh/uv/install.sh', installer)
    run(['sh', installer])
if not REPO.exists():
    run(['git', 'clone', 'https://github.com/Aratako/Irodori-TTS.git', REPO])
run(['git', '-C', REPO, 'checkout', '--detach', CODE_REVISION])
run([uv, 'python', 'install', '3.12'])
run([uv, 'sync', '--frozen', '--extra', 'cu128', '--python', '3.12'], cwd=REPO)
PY = REPO / '.venv/bin/python'
probe = write_script('probe_gpu.py', """import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA를 사용할 수 없습니다.')
print('GPU:', torch.cuda.get_device_name(0))
""")
run([PY, probe], cwd=REPO)
validate = write_script('validate_adapter.py', """import sys
from safetensors import safe_open
with safe_open(sys.argv[1], framework='pt', device='cpu') as weights:
    count = len(list(weights.keys()))
    if not count:
        raise ValueError('LoRA 텐서가 없습니다.')
    print('LoRA 텐서 수:', count)
""")
run([PY, validate, ADAPTER / 'adapter_model.safetensors'], cwd=REPO)
print('남은 로컬 디스크: %.1f GiB' % (shutil.disk_usage(WORK).free / 2**30))

In [ ]:
#@title 4. RF 기본 모델 다운로드 (로컬 캐시)
download = write_script('download_model.py', """import sys
from huggingface_hub import snapshot_download
snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], local_dir=sys.argv[3],
                  allow_patterns=["model.safetensors", "tokenizer/*", "README.md"])
""")
run([PY, download, MODEL_ID, MODEL_REVISION, MODEL_DIR], cwd=REPO)
BASE = MODEL_DIR / 'model.safetensors'
assert BASE.is_file()

In [ ]:
#@title 5. 추론 화면 시작 → 공유 링크
if globals().get('app_proc') is not None and app_proc.poll() is None:
    raise RuntimeError('서버가 실행 중입니다. 다시 시작하려면 6번으로 종료하세요.')
if not globals().get('ADAPTER') or not all((ADAPTER / name).is_file() for name in ADAPTER_FILES):
    raise RuntimeError('2번에서 LoRA를 먼저 가져오세요.')
if not globals().get('BASE') or not BASE.is_file():
    raise RuntimeError('4번에서 기본 모델을 다운로드하세요.')

wrapper = write_script('launch_gradio.py', r"""import os, sys
from pathlib import Path
sys.path.insert(0, os.environ['IRODORI_REPO'])
import gradio_app as app
app._default_checkpoint = lambda: os.environ['IRODORI_BASE']
demo = app.build_ui()
adapters = [c for c in demo.blocks.values()
            if getattr(c, 'label', '') == 'LoRA Adapter Directory (optional)']
if len(adapters) != 1:
    raise RuntimeError('공식 UI의 LoRA 입력란을 찾을 수 없습니다.')
selected = os.environ['IRODORI_ADAPTER']
with demo:
    demo.load(lambda: selected, inputs=None, outputs=adapters[0], queue=False)
demo.queue(default_concurrency_limit=1, max_size=8)
auth = None
if os.environ['IRODORI_LOGIN'] == '1':
    auth = (os.environ.pop('IRODORI_USER'), os.environ.pop('IRODORI_PASSWORD'))
demo.launch(server_name='127.0.0.1', server_port=int(os.environ['IRODORI_PORT']),
    share=True, auth=auth,
    blocked_paths=['/content/drive', str(Path.home() / '.config'),
                   os.environ['HF_HOME'], os.environ['IRODORI_ADAPTER_ROOT'],
                   str(Path(os.environ['IRODORI_BASE']).parent)],
    css=app.EMOJI_PALETTE_CSS)
""")

app_env = os.environ.copy()
app_env.update(IRODORI_REPO=str(REPO), IRODORI_BASE=str(BASE),
    IRODORI_ADAPTER=str(ADAPTER), IRODORI_ADAPTER_ROOT=str(ADAPTER_ROOT),
    IRODORI_PORT=str(server_port), IRODORI_LOGIN='1' if enable_login else '0')
if enable_login:
    password = getpass.getpass('Gradio 로그인 비밀번호 (8자 이상): ')
    if len(password) < 8:
        del password
        raise ValueError('비밀번호는 8자 이상 입력하세요.')
    app_env.update(IRODORI_USER=gradio_username, IRODORI_PASSWORD=password)
    del password
app_log = WORK / 'gradio.log'
try:
    with app_log.open('w', encoding='utf-8') as stream:
        app_proc = subprocess.Popen([str(PY), '-u', str(wrapper)], cwd=REPO, env=app_env,
            stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
finally:
    app_env.pop('IRODORI_PASSWORD', None)

try:
    deadline = time.monotonic() + 180
    public_url = None
    while time.monotonic() < deadline:
        if app_proc.poll() is not None:
            raise RuntimeError(app_log.read_text(encoding='utf-8', errors='replace')[-8000:])
        log = app_log.read_text(encoding='utf-8', errors='replace')
        found = re.search(r'https://[a-zA-Z0-9-]+\.gradio\.live', log)
        ready = False
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{server_port}/', timeout=2) as response:
                ready = response.status == 200
        except urllib.error.HTTPError as error:
            ready = error.code in (401, 403)
        except (urllib.error.URLError, TimeoutError):
            pass
        if ready and found:
            public_url = found.group(0)
            break
        time.sleep(1)
    if not public_url:
        raise RuntimeError('공유 링크를 만들지 못했습니다. 잠시 후 5번을 다시 실행하세요.\n' +
                           app_log.read_text(encoding='utf-8', errors='replace')[-4000:])
except BaseException:
    stop_process(app_proc)
    raise

from IPython.display import display, Markdown
display(Markdown(f'**추론 화면 열기:** [{public_url}]({public_url})'))
print('로그인 ID: ' + gradio_username if enable_login else '로그인 없이 접속합니다.')
print('LoRA Adapter Directory 자동 입력:', ADAPTER)
print('WAV 저장 위치:', REPO / 'gradio_outputs')
print('참조 WAV 업로드 → 일본어 문장 입력 → Generate')

**음성 생성:** 공유 링크를 연 뒤 `Advanced (Optional)`의 `LoRA Adapter Directory`에 위 경로가 들어 있는지 확인하세요.
`Reference Audio`에 참조 WAV를 올리고 `Text`에 일본어 문장을 입력한 뒤 **Generate**를 누릅니다.
기본 모델과 음성 코덱은 첫 생성 시 로드되므로 첫 요청은 시간이 걸립니다. 생성 결과의 다운로드 버튼으로 WAV를 PC에 보관하세요.

**LoRA 바꾸기:** 6번에서 종료 → 1번 경로 변경 → 2번 → 5번 순서로 실행합니다. 같은 런타임에서는 3~4번 설치·다운로드를 반복할 필요가 없습니다.
메모리 부족 시 `Num Candidates`를 1로 유지하고 짧은 참조 WAV와 문장으로 확인하세요.

**실행 범위:** ZIP·폴더 입력과 노트북 코드 구조는 로컬에서 확인하며, Colab GPU에서의 실제 음성 생성은 별도 확인이 필요합니다.

In [ ]:
#@title 6. 추론 서버 종료 (사용을 마친 뒤 선택 실행)
stop_server = False #@param {type:"boolean"}
if stop_server:
    stop_process(globals().get('app_proc'))
    print('추론 서버 종료 완료. 로컬 LoRA와 생성 WAV는 유지됩니다.')
else:
    print('서버를 종료하려면 stop_server를 체크하고 이 셀을 실행하세요.')